### This EXTRACT_CORPUS notebook 

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publication date and type (articles, etc)


In [37]:
%run common_setup.ipynb

In [38]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return

    # def _extract_works_by_journal(self):
    #     # Extract OA works for the journal set, for publication years 2010+ to now
    #     self._match_journals()
    #     hold = []
    #     for row in self.journals.itertuples():
    #         if row.Index > 64:
    #             continue
    #         source_id = row.source_id
    #         if source_id is None:
    #             print(f'source_is IS None FOR {row= }')
    #             continue
    #         reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
    #         if isinstance(oa := self._cache_manager(task=reader), pd.DataFrame) and len(oa) > 0:
    #             oa = self._filter_works(works=oa)
    #             self._sql_appender(df=oa, row=row.Index)
    #             print(f'APPENDED {row.Index = } {oa.shape = }')
    #         else:
    #             print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
    #         if row.Index % 1 == 0:
    #             print(f'{row.Index}/{len(self.journals)} completed')             
    #     return

    def extract_works_by_journal(self):

        sql = """
                -- ETL TO EXTRACT works FOR journal list
                -- ======================================
                CREATE OR REPLACE TABLE project.raw AS
                SELECT -- DISTINCT ON (doi)
                        w.*,
                        lower(title) AS lower_title,
                        IF (first_page = last_page, 0, try_cast(last_page AS INT) - try_cast(first_page AS INT)) AS page_count
                FROM project.jcr_matches
                LEFT JOIN works.works w
                USING (source_id)
                WHERE publication_year > 2009 AND publication_year < 2025
                        AND (is_retracted = false OR is_retracted IS NULL)
                        AND is_paratext = false
                        -- AND (first_page IS NULL OR page_count > 1)
                        -- AND doi IS NOT NULL
                        -- AND referenced_works_count != 0
                        AND contains(lower_title, 'editor') = false 
                        AND contains(lower_title, 'issue information') = false
                        AND contains(lower_title, 'index') = false
                        AND contains(lower_title, 'book review') = false
                        AND contains(lower_title, 'isbn') = false
                        AND contains(lower_title, 'calendar of events') = false
                        AND contains(lower_title, 'notes on contributors') = false
                        AND regexp_matches(lower_title, '^announcements$') = false
                        AND regexp_matches(lower_title, '^acknowledgement') = false
                        AND regexp_matches(lower_title, '^vol[.u ] ') = false
                        AND regexp_matches(lower_title, '^focus on authors$') = false
                        AND regexp_matches(lower_title, '^publications received$') = false
                        AND list_contains(['article', 'review', 'letter'], w."type") = true
                -- ORDER BY page_count
        """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.raw").show()
        self.db.sql("SELECT * FROM project.raw").show()
        return
 
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return

In [39]:
class ExtractAuthorshipsReferencesTopics(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def authorships_etl(self):
        print("authorships")
        sql = """
            -- ETL TO EXTRACT authorships FOR project
            -- ======================================
            CREATE OR REPLACE TABLE project.authorships AS
            SELECT work_id,
                    author_id,
                    author_name,
                    institution.id AS institution_id,
                    institution.display_name AS institution_name,
                    institution.country_code AS country_code
                FROM
                (SELECT id AS work_id,
                        author.id AS author_id,
                        author.display_name AS author_name,
                        unnest(institutions) AS institution
                    FROM project.raw
                    INNER JOIN works.authorships
                    ON id = work_id)
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(DISTINCT work_id), count(DISTINCT author_id), count(DISTINCT institution_id) FROM project.authorships").show()
        return
    
    def references_etl(self):
        print("references")
        sql = """
            -- ETL FOR endogenous citer_cited
            -- =============================================
            CREATE OR REPLACE TABLE project.citer_cited AS
                WITH
                citer_cited_CTE AS
                    (SELECT id AS citer_id,
                            publication_year AS citer_year,
                            unnest(referenced_works) AS cited_id
                        FROM project.raw
                    ),
                citer_cited_filtered_CTE AS
                    (SELECT DISTINCT citer_id,
                            citer_year,
                            cited_id,
                            publication_year AS cited_year
                        FROM citer_cited_CTE
                        INNER JOIN project.raw
                        ON cited_id = id
                    )
            
            SELECT *,
                    cited_year - citer_year - 1 AS delta_t
                FROM citer_cited_filtered_CTE
                WHERE delta_t <= 0
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*), count(DISTINCT citer_id), count(DISTINCT cited_id) FROM project.citer_cited").show()
        return
    
    def topics_etl(self):
        print("topics")
        sql = """
            -- ETL FOR topics MATCHES raw.work_id to works.topics
            -- ==================================================
            CREATE OR REPLACE TABLE project.topics AS
            SELECT DISTINCT r.id AS work_id,
                    topic_score,
                    topic_id,
                    display_name AS topic_name,
                    subfield.id AS subfield_id,
                    subfield.display_name AS subfield_name,
                    field.id AS field_id,
                    field.display_name AS field_name,
                    domain.id AS domain_id,
                    domain.display_name AS dmoain_name
                FROM project.raw r
                INNER JOIN works.topics w
                ON r.id = w.work_id
                LEFT JOIN topics.topics t
                ON t.id = topic_id
            """
        self.db.sql(sql)
        self.db.sql("""SELECT count(*), 
                                count(DISTINCT work_id), count(DISTINCT topic_id), count(DISTINCT subfield_id),
                                count(DISTINCT field_id), count(DISTINCT domain_id)  FROM project.topics""").show()
        return

In [40]:
class  ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        sql = """
            -- ETL authors FROM raw.authorships JOIN authors.authors
            -- =====================================================
            CREATE OR REPLACE TABLE project.authors AS
            SELECT DISTINCT author_id,
                    author_name,
                    orcid,
                    display_name_alternatives,
                    summary_stats.works_count AS works_count, 
                    summary_stats.cited_by_count AS cited_by_count, 
                    summary_stats.h_index AS h_index,
                    summary_stats."2yr_h_index" AS h_index_2yr
                FROM project.authorships a
                INNER JOIN authors.authors aa
                ON aa.id = a.author_id
            """
        self.db.sql(sql)
        df = self.db.sql("SELECT * FROM project.authors").df()
        self._load_authors(df=df)
        return
    
    def _load_authors(self, df=None):
        df[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in df.author_name]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("SELECT * FROM project.authors").show()
        self.db.sql("SELECT count(*) FROM project.authors").show()
        return
    
    def duplicate_db_as_backup(self):
        self.db.sql("BEGIN TRANSACTION; COPY FROM DATABASE project TO backup; COMMIT;")
        return         

In [41]:
def main():

    # jetl = ArticlesETL()
    # jetl.extract_works_by_journal()
    # print(jetl.db.sql("DESCRIBE TABLE project.raw").df())
    # jetl.db.sql("SELECT * FROM project.raw").show()
    # sql = """SELECT count(DISTINCT source_id) FROM project.raw"""
    # jetl.db.sql(sql).show()
    # # # jetl.duplicate_db_as_backup()
    # jetl.db.close()

    # ea = ExtractAuthorshipsReferencesTopics()
    # ea.authorships_etl()
    # ea.references_etl()
    # ea.topics_etl()
    # ea.db.close()

    eauthors = ExtractAuthors()
    eauthors.extract_authors()
    # eauthors.duplicate_db_as_backup()
    eauthors.db.close()
    
    # mdas = MatchDomingoAuthorSample()
    # mdas.extract_sample()
    # mdas.match_sample()
    # mdas.load_sample()
    # mdas.db.close()

    # mdss = MatchDomingoSourceSample()
    # mdss.special_issn()
    # mdss.extract_jcr()
    # mdss.extract_journals()
    # mdss.match_sources()
    # mdss.db.close()

In [42]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ authors              │ [author_id, author…  │ [VARCHAR, VARCHAR, 'VARCHAR[]',…  │ false     │
│ project      │ main    │ autho